<a href="https://colab.research.google.com/github/jayeshpanwar17/Solar-Panel-detection-from-satellite-images/blob/main/solarpaneldetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install the ultralytics library
!pip install ultralytics -q

# 2. Unzip your dataset into the cloud environment
!unrar x /content/data300.rar /content/data300/

print("Environment setup and data extraction complete!")


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/data300.rar

Creating    /content/data300                                          OK
Creating    /content/data300/data300                                  OK
Extracting  /content/data300/data300/data(1).yaml                          0%  OK 
Extracting  /content/data300/data300/data(2).yaml                          0%  OK 
Extracting  /content/data300/data300/data.yaml                             0%  OK 
Extracting  /content/data300/data300/README.dataset.txt                    0%  OK 
Extracting  /content/data300/data300/README.roboflow.txt                   0%  OK 
Creating    /content/data300/data300/test                             OK
Creating    /content/data300/data300/test/images                      OK
Extracting  /content/data300/data300/test/images/tile_0_1920_png.rf.yHVlTVIJ2fZqdKq5Wgct.png       0%  OK 
Extracting  /conte

In [ ]:
import yaml

yaml_path = '/content/data300/data300/data.yaml'

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Update paths to match the exact nested structure
data['train'] = '/content/data300/data300/train/images'
data['val'] = '/content/data300/data300/valid/images'
data['test'] = '/content/data300/data300/test/images'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("data.yaml paths updated successfully with the correct nested structure!")

data.yaml paths updated successfully with the correct nested structure!


In [ ]:
from ultralytics import YOLO

# 1. Load the pre-trained nano backbone
model = YOLO("yolov8s.pt")

# 2. Train the model using the GPU
results = model.train(
    data="/content/data300/data300/data.yaml",
    epochs=50,                           # Increased to let your loss curves mature
    patience=0,                          # Stops automatically if metrics plateau for 50 epochs
    imgsz=640,                            # Standard image resolution
    batch=16,                             # Safe batch size for T4 GPU memory
    device=0,                             # Explicitly forces the hosted GPU usage

    # --- Spatial Augmentations to boost your 223 image count ---
    degrees=15.0,                         # Slight random rotations
    flipud=0.5,                           # Vertical flips (great for aerial/satellite panel views)
    fliplr=0.5,                           # Horizontal flips
    scale=0.5,                            # Random zoom-in/out to handle varied camera heights
    mosaic=1.0,                           # Combines images into collages to force pattern learning
    mixup=0.1                             # Overlays images to help with tightly packed arrays
)

print("\n--- Training Complete! Evaluating performance metrics... ---")

# 3. Formally validate the optimized model weights

best_model_path = f"{results.save_dir}/weights/best.pt"
metrics = YOLO(best_model_path).val()
print(f"Optimized mAP50-95: {metrics.box.map * 100:.2f}%")
print(f"Optimized Precision: {metrics.box.mp * 100:.2f}%")
print(f"Optimized Recall: {metrics.box.mr * 100:.2f}%")

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data300/data300/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pat

In [ ]:
from ultralytics import YOLO

# 1. Load the pre-trained nano backbone
model = YOLO("yolov8s.pt")

# 2. Train the model using the GPU
results = model.train(
    data="/content/data300/data.yaml",
    epochs=300,                           # Increased to let your loss curves mature
    patience=0,                          # Stops automatically if metrics plateau for 50 epochs
    imgsz=640,                            # Standard image resolution
    batch=16,                             # Safe batch size for T4 GPU memory
    device=0,                             # Explicitly forces the hosted GPU usage

    # --- Spatial Augmentations to boost your 223 image count ---
    degrees=15.0,                         # Slight random rotations
    flipud=0.5,                           # Vertical flips (great for aerial/satellite panel views)
    fliplr=0.5,                           # Horizontal flips
    scale=0.5,                            # Random zoom-in/out to handle varied camera heights
    mosaic=1.0,                           # Combines images into collages to force pattern learning
    mixup=0.1                             # Overlays images to help with tightly packed arrays
)

print("\n--- Training Complete! Evaluating performance metrics... ---")

# 3. Formally validate the optimized model weights

best_model_path = f"{results.save_dir}/weights/best.pt"
metrics = YOLO(best_model_path).val()
print(f"Optimized mAP50-95: {metrics.box.map * 100:.2f}%")
print(f"Optimized Precision: {metrics.box.mp * 100:.2f}%")
print(f"Optimized Recall: {metrics.box.mr * 100:.2f}%")